In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def make_student_success(n=600, seed=1):
    rng = np.random.default_rng(seed)

    X = pd.DataFrame({
        "hours": rng.integers(0, 12, n),
        "attendance": rng.integers(40, 100, n),
        "sleep": rng.integers(4, 9, n),
        "practice_tests": rng.integers(0, 6, n),
    })

    y = (((X["hours"] >= 5) &
          (X["attendance"] >= 70) &
          (X["sleep"] >= 6))
          |
          (X["practice_tests"] >= 4)).astype(int)

    y = np.where(rng.random(n) < 0.07, 1-y, y)
    return X, y

X, y = make_student_success()

# ✅ Stratify keeps class balance similar in train/test (critical in classification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=3,
    stratify=y
)

# ✅ max_iter avoids "did not converge" issues as datasets get bigger
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
print("\nReport:\n", classification_report(y_test, pred))

# ✅ Use DataFrame with column names to avoid feature-order mistakes
new_student = pd.DataFrame([{
    "hours": 6,
    "attendance": 80,
    "sleep": 7,
    "practice_tests": 0
}])

print("Prediction:", int(model.predict(new_student)[0]))



# What is `stratify=y` and Why It Matters

This is **very important in classification problems**.

## The Problem Without Stratify

Suppose your dataset looks like this:

70% students pass  
30% students fail  

If we split randomly **without stratify**, the test set might become:

85% pass  
15% fail  

Or even worse:

95% pass  
5% fail  

Now your evaluation becomes **distorted**.

A **dumb model** that predicts **"pass" for everyone** could appear extremely accurate.

Example:

Predict: PASS for every student  
Accuracy = 95%

But the model **learned nothing**.

---

# What `stratify=y` Does

When we write:

```python
train_test_split(..., stratify=y)
```

It forces the split to **preserve the class distribution**.

Meaning:

- Training set keeps the same balance
- Test set keeps the same balance

If the original dataset is:

70% pass  
30% fail  

Then after the split:

Train ≈ 70% pass / 30% fail  
Test ≈ 70% pass / 30% fail  

The **class balance is preserved**.

This makes the evaluation **fair and reliable**.

---

# Simple Real-World Analogy

Imagine you are building a **cancer detection model**.

Dataset distribution:

10% cancer cases  
90% healthy cases  

If the test set accidentally contains:

2% cancer  
98% healthy  

Your evaluation becomes **meaningless**.

The model could predict **"healthy" for everyone** and still look extremely accurate.

`stratify` prevents this problem by ensuring the **test data reflects reality**.

---

# Key Idea to Remember

Good ML evaluation requires:

- Multiple splits
- Balanced data
- Fair testing

That is exactly what:

```
random_state
+
stratify=y
```

helps achieve.